In [0]:
# Data Quality Checks Framework (Enterprise Scale)
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, when, current_timestamp, lit,
    sum as spark_sum
)
import datetime

spark = SparkSession.builder.getOrCreate()

CATALOG_SCHEMA = "ecomstore.ecomlake"
DQ_LOG_TABLE = f"{CATALOG_SCHEMA}.gold_dq_log"

def log_dq_result(table_name, check_name, check_description, failed_records):
    """Standardized DQ check logger"""
    status = "PASS" if failed_records == 0 else "WARNING"
    log_row = spark.createDataFrame([{
        "check_id":         f"{table_name}_{check_name}_{datetime.date.today()}",
        "table_name":       table_name,
        "check_name":       check_name,
        "check_description":check_description,
        "failed_records":   int(failed_records),
        "status":           status,
        "run_timestamp":    datetime.datetime.now().isoformat(),
        "pipeline_run_date":str(datetime.date.today())
    }])
    log_row.write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(DQ_LOG_TABLE)


# 1. QUARANTINE PIPELINE CHECKS (Dynamic Loop)
# This dictionary makes the DQ framework infinitely scalable
quarantine_tables = {
    "silver_quarantine_orders":      "Orders rejected due to structural or geo faults",
    "silver_quarantine_inventory":   "Inventory rejected due to null IDs or quantities",
    "silver_quarantine_order_items": "Order items rejected due to missing IDs or prices",
    "silver_quarantine_payments":    "Payments rejected due to missing IDs or amounts",
    "silver_quarantine_returns":     "Returns rejected due to missing IDs or amounts",
    "silver_quarantine_shipments":   "Shipments rejected due to missing IDs or dates"
}

print("🛡️ Sweeping Quarantine Tables...")
for q_table, description in quarantine_tables.items():
    full_table_path = f"{CATALOG_SCHEMA}.{q_table}"
    
    # Failsafe: Only count if the table has actually been created by the Silver layer
    if spark.catalog.tableExists(full_table_path):
        failed_count = spark.read.table(full_table_path).count()
    else:
        failed_count = 0
        
    log_dq_result(q_table, "quarantine_volume", description, failed_count)


# 2. SINGLE-PASS TABLE CHECKS

print("⚡ Running Single-Pass Table Rule Checks...")

# A. Orders Table Rules
orders = spark.read.table(f"{CATALOG_SCHEMA}.silver_orders")
valid_statuses = ["placed", "confirmed", "processing", "shipped", "out_for_delivery", "delivered", "cancelled", "returned"]

orders_dq = orders.agg(
    spark_sum(when(col("order_id").isNull(), 1).otherwise(0)).alias("null_order_id"),
    spark_sum(when(col("customer_id").isNull(), 1).otherwise(0)).alias("null_customer_id"),
    spark_sum(when(~col("status").isin(valid_statuses), 1).otherwise(0)).alias("invalid_status"),
    spark_sum(when(col("final_amount") < 0, 1).otherwise(0)).alias("negative_amounts"),
    spark_sum(when(col("order_date") > current_timestamp(), 1).otherwise(0)).alias("future_order_dates")
).collect()[0]

log_dq_result("silver_orders", "null_order_id", "Non-null order_id", orders_dq["null_order_id"])
log_dq_result("silver_orders", "null_customer_id", "Non-null customer_id", orders_dq["null_customer_id"])
log_dq_result("silver_orders", "invalid_status", "Valid order status", orders_dq["invalid_status"])
log_dq_result("silver_orders", "negative_amounts", "Positive final amount", orders_dq["negative_amounts"])
log_dq_result("silver_orders", "future_order_dates", "No future order dates", orders_dq["future_order_dates"])


# B. Customers SCD Table Rules
customers = spark.read.table(f"{CATALOG_SCHEMA}.silver_customers_scd").filter(col("is_current") == True)
customers_dq = customers.agg(
    spark_sum(when(col("email").isNotNull() & ~col("email").rlike(r".*@.*\..*"), 1).otherwise(0)).alias("invalid_email"),
    spark_sum(when(~col("customer_tier").isin("Bronze","Silver","Gold","Platinum"), 1).otherwise(0)).alias("invalid_tier")
).collect()[0]

log_dq_result("silver_customers_scd", "invalid_email_format", "Valid email regex", customers_dq["invalid_email"])
log_dq_result("silver_customers_scd", "invalid_tier", "Standard customer tier", customers_dq["invalid_tier"])


# C. Products Table Rules
products = spark.read.table(f"{CATALOG_SCHEMA}.silver_products")
prod_dq = products.agg(
    spark_sum(when(col("selling_price") > col("mrp"), 1).otherwise(0)).alias("price_logic"),
    spark_sum(when(col("category").isNull(), 1).otherwise(0)).alias("null_category")
).collect()[0]

log_dq_result("silver_products", "price_logic", "Selling price <= MRP", prod_dq["price_logic"])
log_dq_result("silver_products", "completeness", "Non-null category", prod_dq["null_category"])


# 3. DQ Summary Dashboard
print("\n📊 DQ Summary for today:")
spark.sql(f"""
    SELECT table_name, check_name, status, failed_records, run_timestamp
    FROM {DQ_LOG_TABLE}
    WHERE pipeline_run_date = current_date()
    ORDER BY status DESC, table_name
""").show(n=50, truncate=False)